# House Price Prediction - Inference

Load trained model and make predictions on new data.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import pickle

sys.path.insert(0, str(Path.cwd().parent.parent))
from pipeline.supabase_handler import fetch_csv_from_supabase

print("✅ Imports successful")

## Step 1: Load Trained Model

In [ ]:
# Find and load latest model
model_dir = Path('saved_models')
model_files = sorted(model_dir.glob('*.joblib'))

if not model_files:
    print("❌ No trained models found!")
    print("Run training first: python train_simple.py")
else:
    latest_model = model_files[-1]
    model_name = latest_model.stem
    
    print(f"Loading model: {model_name}")
    model = joblib.load(latest_model)
    
    # Load metadata
    meta_path = model_dir / f"{model_name}_meta.pkl"
    with open(meta_path, 'rb') as f:
        metadata = pickle.load(f)
    
    print(f"✓ Model loaded")
    print(f"  Type: {metadata['model_type']}")
    print(f"  R² Score: {metadata['metrics']['r2_score']:.4f}")
    print(f"  RMSE: {metadata['metrics']['rmse']:.4f} billion VND")

## Step 2: Load Data from Supabase

In [ ]:
print("\nLoading data from Supabase...")
df = fetch_csv_from_supabase("Raw_Features")

# Convert price
df['price_billion_vnd'] = df['price_vnd'] / 1e9

print(f"✓ Loaded {len(df)} records")

## Step 3: Prepare Features

In [ ]:
# Get features from metadata
features = metadata['features']

# Prepare feature matrix
X = df[features].copy()
X = X.fillna(X.median())

print(f"Features: {len(features)}")
print(f"X shape: {X.shape}")

## Step 4: Make Predictions

In [ ]:
print("Making predictions...")
y_pred = model.predict(X)
df['predicted_price_billion_vnd'] = y_pred

print(f"✓ Made {len(y_pred)} predictions")
print(f"\nPrediction range: {y_pred.min():.2f} - {y_pred.max():.2f} billion VND")

## Step 5: Evaluate Predictions

In [ ]:
# Calculate errors if actual prices available
if 'price_billion_vnd' in df.columns:
    df['error_billion_vnd'] = df['predicted_price_billion_vnd'] - df['price_billion_vnd']
    mae = np.abs(df['error_billion_vnd']).mean()
    rmse = np.sqrt((df['error_billion_vnd'] ** 2).mean())
    
    print("Prediction Error Statistics:")
    print(f"  MAE: {mae:.4f} billion VND")
    print(f"  RMSE: {rmse:.4f} billion VND")
    print(f"  Mean Error: {df['error_billion_vnd'].mean():.4f} billion VND")
    
    # Sample predictions
    print("\nSample predictions (first 10):")
    sample = df[['area_m2', 'distance_to_center_km', 'predicted_price_billion_vnd', 'price_billion_vnd', 'error_billion_vnd']].head(10)
    print(sample.to_string(index=False))

## Step 6: Visualize Results

In [ ]:
if 'price_billion_vnd' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Actual vs Predicted
    axes[0].scatter(df['price_billion_vnd'], df['predicted_price_billion_vnd'], alpha=0.5)
    min_price = min(df['price_billion_vnd'].min(), df['predicted_price_billion_vnd'].min())
    max_price = max(df['price_billion_vnd'].max(), df['predicted_price_billion_vnd'].max())
    axes[0].plot([min_price, max_price], [min_price, max_price], 'r--', lw=2)
    axes[0].set_xlabel('Actual Price (billion VND)')
    axes[0].set_ylabel('Predicted Price (billion VND)')
    axes[0].set_title('Actual vs Predicted Prices')
    axes[0].grid(True, alpha=0.3)
    
    # Error distribution
    axes[1].hist(df['error_billion_vnd'], bins=30, edgecolor='black', alpha=0.7)
    axes[1].axvline(df['error_billion_vnd'].mean(), color='r', linestyle='--', 
                    label=f"Mean: {df['error_billion_vnd'].mean():.2f}")
    axes[1].set_xlabel('Prediction Error (billion VND)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of Errors')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Step 7: Save Predictions

In [ ]:
# Save predictions
pred_dir = Path('../..') / 'data' / 'predictions'
pred_dir.mkdir(parents=True, exist_ok=True)

output_file = pred_dir / 'predictions_latest.csv'
df.to_csv(output_file, index=False)

print(f"✓ Saved predictions: {output_file}")
print(f"  Records: {len(df)}")
print(f"  Size: {output_file.stat().st_size / 1024 / 1024:.2f} MB")

print(f"\n✅ Inference complete!")